In [1]:
!pip install datasets -q
from datasets import load_dataset

stream = load_dataset("smartcat/Amazon-2023-GenQ", split="train", streaming=True)

first = next(iter(stream))
print(list(first.keys()))
print({k: str(v)[:100] for k, v in first.items()})

README.md:   0%|          | 0.00/5.20k [00:00<?, ?B/s]

['parent_asin', 'main_category', 'title', 'description', 'features', 'embellished_description', 'brand', 'images', 'short_query', 'long_query']
{'parent_asin': 'B016UIUCRG', 'main_category': 'Sports & Outdoors', 'title': 'NHL Molded Auto Emblem', 'description': 'Showcase your team spirit with this eye-catching Chrome Finished Auto Emblem 3D Sticker by Rico Indu', 'features': 'Made in US\nEye-catching auto emblem measures 5-inches by 5-inches\nDecorated with dynamic and bold te', 'embellished_description': '###Description###\nShowcase your team spirit with the NHL Molded Auto Emblem 3D Sticker by Rico Indus', 'brand': 'Rico Industries', 'images': "[{'hi_res': 'https://m.media-amazon.com/images/I/71HKBZL3H+L._AC_UL1500_.jpg', 'large': 'https://m.m", 'short_query': 'NHL auto emblem sticker', 'long_query': '3D molded NHL auto emblem sticker for Washington Capitals'}


In [2]:
import pandas as pd
from datasets import load_dataset

stream = load_dataset("smartcat/Amazon-2023-GenQ", split="train", streaming=True)
stream = stream.shuffle(seed=42, buffer_size=10000)

rows = []
for row in stream:
    rows.append({"parent_asin": row["parent_asin"], "main_category": row["main_category"],
                 "title": row["title"], "brand": row["brand"],
                 "description": row["description"], "features": row["features"]})
    if len(rows) == 40000:
        break

df = pd.DataFrame(rows)
print(df.shape)
print(df["main_category"].value_counts(dropna=False))
print(df.isnull().sum())

(40000, 6)
main_category
AMAZON FASHION               26350
All Beauty                    5532
Sports & Outdoors             3576
Unknown                       2238
Health & Personal Care         803
Amazon Home                    525
Tools & Home Improvement       304
Automotive                     151
Industrial & Scientific        108
Office Products                 75
Pet Supplies                    60
Toys & Games                    57
Arts, Crafts & Sewing           53
Cell Phones & Accessories       46
All Electronics                 34
Baby                            31
Premium Beauty                  30
Grocery                         27
Name: count, dtype: int64
parent_asin      0
main_category    0
title            0
brand            0
description      0
features         0
dtype: int64


In [3]:
fashion = df[df["main_category"] == "AMAZON FASHION"].copy()
print("Fashion products:", len(fashion))

cols = ["title", "brand", "description", "features"]
fashion[cols] = fashion[cols].astype(str).replace({"None": "", "nan": "", "[]": "", "{}": ""})

print("Empty descriptions:", (fashion["description"].str.len() < 3).sum())
print("Empty features:", (fashion["features"].str.len() < 3).sum())
print("Empty brands:", (fashion["brand"].str.len() < 2).sum())

fashion = fashion[fashion["title"].str.len() > 3]
fashion = fashion.drop_duplicates(subset="title").reset_index(drop=True)
print("After removing short and duplicate titles:", len(fashion))

fashion["text_basic"] = fashion["title"] + " " + fashion["brand"]
fashion["text_full"] = (fashion["title"] + " " + fashion["brand"] + " "
                        + fashion["features"] + " " + fashion["description"])

clean = lambda s: (s.str.lower().str.replace(r"[^a-z0-9 ]", " ", regex=True)
                   .str.replace(r"\s+", " ", regex=True).str.strip())
fashion["text_basic"] = clean(fashion["text_basic"])
fashion["text_full"] = clean(fashion["text_full"])

print(fashion[["title", "brand"]].head(8))
print(fashion["text_full"].iloc[0][:300])
print(fashion["text_full"].str.split().str.len().describe())

Fashion products: 26350
Empty descriptions: 20
Empty features: 1018
Empty brands: 90
After removing short and duplicate titles: 25994
                                               title  \
0  AEROPOSTALE Aéropostale Women's Quartz Metal R...   
1  Be Kind Tank Tops Women Funny Letter Graphic T...   
2  Siteshy 70th Birthday Gifts for Women Men Vint...   
3      Van Halen Men's Classic Red Logo T-Shirt Blue   
4  QUEEN PLUS Girls Casual Maxi Floral Dress Long...   
5         Tommy Hilfiger Women's Julie3 Ankle Bootie   
6  Readymade Simply Classy Printed Cotton Salwar ...   
7  NCAA Teams Mens Pajamas - Mens Pants Mens Shir...   

                            brand  
0  A Classic Time Watch Co., Inc.  
1                        KIMSOONG  
2                         Siteshy  
3                    PLASTIC HEAD  
4                      QUEEN PLUS  
5                  Tommy Hilfiger  
6                        ladyline  
7                 Profile Varsity  
aeropostale a ropostale women s quart

In [4]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

tfidf_full = TfidfVectorizer(max_features=30000, ngram_range=(1, 2), min_df=3,
                             max_df=0.5, stop_words="english", sublinear_tf=True)
X_full = tfidf_full.fit_transform(fashion["text_full"])

tfidf_basic = TfidfVectorizer(ngram_range=(1, 2), min_df=2,
                              stop_words="english", sublinear_tf=True)
X_basic = tfidf_basic.fit_transform(fashion["text_basic"])

print(X_full.shape, X_basic.shape)

def show(index, X, n=5):
    scores = linear_kernel(X[index], X).ravel()
    scores[index] = -1
    top = np.argsort(scores)[::-1][:n]
    for i in top:
        print("  ", round(scores[i], 3), fashion["title"][i][:70])

for idx in [3, 4, 5]:
    print("PRODUCT:", fashion["title"][idx][:70])
    print(" Full text version:")
    show(idx, X_full)
    print(" Short text version:")
    show(idx, X_basic)
    print()

(25994, 30000) (25994, 42821)
PRODUCT: Van Halen Men's Classic Red Logo T-Shirt Blue
 Full text version:
   0.46 Trevco Men's Kiss Short Sleeve T-Shirt
   0.42 Global Iron Maiden Unisex Distressed Logo T-Shirt
   0.392 Nsync Men's Split Photo T-Shirt White
   0.362 Rolling Stones Men's Est 62 T-Shirt Black
   0.279 Ozzy Osbourne 'Patient No. 9 Gold Logo' (Black) T-Shirt
 Short text version:
   0.266 Disney T-shirt
   0.254 Spira Men's Classic Walker 2
   0.249 HEAD
   0.237 RokFit Men's Original RF Logo Shirt
   0.216 10 PACK: Womens / Girls Assorted Plastic Head & Hair Bands / Hair Orna

PRODUCT: QUEEN PLUS Girls Casual Maxi Floral Dress Long Sleeve Pocket Dresses 5
 Full text version:
   0.292 Rela Bota Women's Sexy Vintage 2 Piece Outfit Ruffle Off Shoulder Crop
   0.269 Women's O Neck Feather Print Shirt Casual Cutout Sleeve Top Blouse(S-3
   0.256 QiJu Sexy Women Girl Lady Swimsuit Swimwear Padded Bikini Push Up
   0.253 Just for Plus Women's Denim Destroyed Ankle Length Skinny Je

In [5]:
t = fashion["title"].str.lower()
conditions = [
    t.str.contains(r"\bwatch(?:es)?\b"),
    t.str.contains(r"\bdress(?:es)?\b|\bgown\b"),
    t.str.contains(r"t-shirt|\btshirt|\btee\b"),
    t.str.contains(r"\bhoodie|\bsweatshirt|\bpullover"),
    t.str.contains(r"\bjeans\b|\bpants\b|\bleggings\b|\bshorts\b"),
    t.str.contains(r"\bshoes?\b|\bboots?\b|\bbootie|\bsneakers?\b|\bsandals?\b|\bheels?\b"),
    t.str.contains(r"\bring\b|\bnecklace\b|\bearrings?\b|\bbracelet\b"),
    t.str.contains(r"\bsocks?\b"),
    t.str.contains(r"\bswimsuit|\bbikini|\bswimwear"),
    t.str.contains(r"\bjacket\b|\bcoat\b|\bvest\b"),
    t.str.contains(r"\bbra\b|\bunderwear|\bpanties|\blingerie"),
    t.str.contains(r"\bhat\b|\bcap\b|\bbeanie"),
]
names = ["watch", "dress", "t-shirt", "hoodie", "pants", "shoes", "jewelry",
         "socks", "swimwear", "jacket", "underwear", "hat"]
fashion["ptype"] = np.select(conditions, names, default="other")
print(fashion["ptype"].value_counts())

labeled = fashion.index[fashion["ptype"] != "other"].values
shares = fashion.loc[labeled, "ptype"].value_counts(normalize=True)
print("Random guess precision:", round((shares ** 2).sum(), 3))

rng = np.random.default_rng(42)
sample = rng.choice(labeled, size=1000, replace=False)
ptype = fashion["ptype"].values

def precision_at_5(sim_function):
    hits = 0
    for start in range(0, len(sample), 200):
        batch = sample[start:start + 200]
        S = sim_function(batch)
        S[np.arange(len(batch)), batch] = -1
        top = np.argsort(-S, axis=1)[:, :5]
        hits += (ptype[top] == ptype[batch][:, None]).sum()
    return hits / (len(sample) * 5)

sim_full = lambda b: linear_kernel(X_full[b], X_full)
sim_basic = lambda b: linear_kernel(X_basic[b], X_basic)
sim_mix = lambda b: 0.5 * sim_full(b) + 0.5 * sim_basic(b)

print("Full text   precision@5:", round(precision_at_5(sim_full), 3))
print("Short text  precision@5:", round(precision_at_5(sim_basic), 3))
print("Mix of both precision@5:", round(precision_at_5(sim_mix), 3))

ptype
other        10594
shoes         3667
jewelry       3078
dress         1421
t-shirt       1377
pants         1347
hat           1070
hoodie         747
jacket         648
watch          636
socks          595
underwear      517
swimwear       297
Name: count, dtype: int64
Random guess precision: 0.134
Full text   precision@5: 0.672
Short text  precision@5: 0.635
Mix of both precision@5: 0.672
